# MSSearch Next-Gen Redesign: Comprehensive Multi-Sample Evaluation Suite (N=30)
## Quantitative Benchmark & Multi-Regime Comparative Diagnostic against Old CVSearch Baseline

This notebook executes an end-to-end multi-sample benchmark across **$N=30$ target samples** from the high-resolution visual search suite (HRBench-8k/4k & V*), comparing four distinct operational regimes:
1. **Direct Full-Canvas Baseline (Old Stage 2)**: Standard VLM direct question answering over downsampled full canvases ($1024\text{px}$).
2. **Old CVSearch Quadtree Baseline**: Rigid $2\times 2$ bisection quadtree, global average-pooled cosine similarity saliency, binary pruning cliff gating ($\tau_{\text{prune}} = 0.25$), and geometric centroid reading.
3. **Redesign Stage 4 Macro Holistic Context**: Elastic Sobel seam-carving, dense spatial token saliency $S^*(p)$, continuous Bayesian evidential consolidation ($E_{\text{post}} = E_{\text{prior}} + \beta c_{\text{VLM}}$), and unified connected-component macro-envelopes.
4. **Redesign Stage 6 Foveated Native 1:1 Peak Detail**: Unscaled native optical resolution crop centered on the Argmax Evidential Peak $p^*$ (eliminating micro-object dilution).
5. **Redesign Unified Engine (Ensemble/Consensus)**: Adaptive selection combining Macro Context and Native 1:1 Peak Detail.

### Artifacts & Deliverables
- **Accuracies**: Global accuracy (%) across all regimes, plus fine-grained subcategory breakdowns (Attribute / Direct vs. Relational / Spatial).
- **Per-Sample Visual Cards**: High-resolution 4-panel visual diagnostic cards saved for every evaluated sample to `outputs/eval_multisource_n30/visualizations/sample_{idx:03d}_card.png`.
- **Global Summary Visualizations**: Publication-grade comparative bar charts, pruning cliff diagnostic plots, and latency Pareto frontiers saved to `outputs/eval_multisource_n30/`.
- **Telemetry Checkpoint**: Streaming JSONL telemetry logs saved to `outputs/eval_multisource_n30/telemetry/`.

In [ ]:
# ==============================================================================
# CELL 1: ENVIRONMENT SETUP, OFFLINE ENFORCEMENT & DUAL-GPU INITIALIZATION
# ==============================================================================
import os
import sys
import gc
import re
import json
import time
import shutil
import random
import warnings
import subprocess
from datetime import datetime
from typing import List, Dict, Tuple, Optional, Any, Set, Union
from dataclasses import dataclass, field

# Enforce strict offline operation for reproducible Kaggle runs
os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"
os.environ["HF_DATASETS_OFFLINE"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

warnings.filterwarnings("ignore")

import torch
import numpy as np
from PIL import Image, ImageDraw, ImageFont
import matplotlib.pyplot as plt
import matplotlib.patches as patches

# Ensure local MSSearch repository is importable
repo_search_paths = [
    "/kaggle/working/MSSearch",
    "/kaggle/working/mysite/docs/experiments/multisource/MSSearch",
    "docs/experiments/multisource/MSSearch",
    "MSSearch",
    os.path.abspath(os.path.join(os.getcwd(), "MSSearch")),
    os.path.abspath(os.path.join(os.getcwd(), "..", "MSSearch"))
]

for p in repo_search_paths:
    if os.path.exists(p) and p not in sys.path:
        sys.path.insert(0, p)
        print(f"[Environment] Added MSSearch repository to sys.path: {p}")
        break

print("=" * 75)
print(f"[Device Topology Check] PyTorch Version: {torch.__version__}")
if torch.cuda.is_available():
    num_gpus = torch.cuda.device_count()
    print(f"[Device Topology Check] Detected {num_gpus} CUDA Device(s):")
    for i in range(num_gpus):
        props = torch.cuda.get_device_properties(i)
        vram_gb = props.total_memory / (1024**3)
        print(f"  - GPU {i}: {props.name} | Total VRAM: {vram_gb:.2f} GB")
    device_vlm = torch.device("cuda:0")
    device_sam = torch.device("cuda:1") if num_gpus > 1 else torch.device("cuda:0")
else:
    print("[Device Topology Check] WARNING: CUDA not detected! Running on CPU fallback.")
    device_vlm = torch.device("cpu")
    device_sam = torch.device("cpu")

print(f"[Device Topology] VLM & SigLIP Device: {device_vlm} | SAM3 Device: {device_sam}")
print("=" * 75)

In [ ]:
# ==============================================================================
# CELL 2: DATASET DISCOVERY & MULTI-SAMPLE (N=30) CANDIDATE POOL SUITE
# ==============================================================================
HRBENCH_SEARCH_DIRS = [
    "D:/Research/VisualSearch/hr-bench_8k",
    "d:/Research/VisualSearch/hr-bench_8k",
    "/kaggle/input/datasets/thnhtt/vs-dataset/hr-bench_8k/mnt/sdb/llp/datasets/zoom_eye_data/zoom_eye_data/hr-bench_8k",
    "/kaggle/input/datasets/thnhtt/vs-dataset/hr-bench_8k",
    "/kaggle/input/datasets/thnhtt/vs-dataset/hr-bench_4k",
    "/kaggle/input/hr-bench-8k",
    "/kaggle/input/hr-bench-4k",
    "docs/experiments/multisource/data/hr-bench_8k",
    "docs/experiments/freeform/ICML26-CVSearch/data/hr-bench_8k"
]

HRBENCH_BASE_DIR = None
for d in HRBENCH_SEARCH_DIRS:
    if os.path.exists(d):
        HRBENCH_BASE_DIR = d
        break

if HRBENCH_BASE_DIR is None:
    raise FileNotFoundError(f"[Dataset Error] Could not find HRBench directory in candidate paths: {HRBENCH_SEARCH_DIRS}")

HRBENCH_ANNOTATION_PATH = os.path.join(HRBENCH_BASE_DIR, "annotation_hr-bench_8k.json")
if not os.path.exists(HRBENCH_ANNOTATION_PATH):
    HRBENCH_ANNOTATION_PATH = os.path.join(HRBENCH_BASE_DIR, "annotation_hr-bench_4k.json")

print(f"[Dataset Discovery] Using HRBench root: {HRBENCH_BASE_DIR}")
print(f"[Dataset Discovery] Loading annotations: {HRBENCH_ANNOTATION_PATH}")

with open(HRBENCH_ANNOTATION_PATH, "r", encoding="utf-8") as f:
    benchmark_dataset = json.load(f)

total_available = len(benchmark_dataset)
print(f"[Dataset Ingestion] Loaded {total_available} total benchmark samples.")

# ------------------------------------------------------------------------------
# BENCHMARK EVALUATION CONFIGURATION KNOBS
# ------------------------------------------------------------------------------
NUM_EVAL_SAMPLES = 30         # Total number of samples to evaluate (e.g. 30)
START_SAMPLE_OFFSET = 0       # Starting index in dataset
CANDIDATE_POOL_SIZE = 3       # N images per sample (1 Target + N-1 Distractors)
DISTRACTOR_WINDOW = 10        # Local sampling window for realistic distractor pool
RANDOM_SEED = 42              # Master deterministic reproducibility seed

OUTPUT_DIR = "outputs/eval_multisource_n30"
VIS_DIR = os.path.join(OUTPUT_DIR, "visualizations")
TELEMETRY_DIR = os.path.join(OUTPUT_DIR, "telemetry")

os.makedirs(VIS_DIR, exist_ok=True)
os.makedirs(TELEMETRY_DIR, exist_ok=True)
print(f"[Output Directories] Initialized evaluation storage:")
print(f"  - Visual Diagnostic Cards: {VIS_DIR}")
print(f"  - Telemetry & Metrics:     {TELEMETRY_DIR}")

# Deterministic Ground-Truth Resolver
def resolve_gt_letter(answer_obj: Any, cycle_idx: int = 0) -> str:
    if isinstance(answer_obj, list):
        if 0 <= cycle_idx < len(answer_obj):
            ans = str(answer_obj[cycle_idx]).strip()
        else:
            ans = str(answer_obj[0]).strip() if answer_obj else ""
    else:
        ans = str(answer_obj).strip()
    m = re.search(r"\(?([A-Za-z])\)?", ans)
    return m.group(1).upper() if m else ans.upper()

def format_options_str(raw_opts: Any) -> str:
    if isinstance(raw_opts, dict):
        return chr(10).join([f"({k}) {v}" for k, v in raw_opts.items()])
    elif isinstance(raw_opts, list):
        formatted = []
        for idx, opt in enumerate(raw_opts):
            s = str(opt).strip()
            if re.match(r"^\(?[A-Za-z]\)?[.: ]", s):
                formatted.append(s)
            else:
                letter = chr(ord("A") + idx)
                formatted.append(f"({letter}) {s}")
        return chr(10).join(formatted)
    return str(raw_opts) if raw_opts else ""

# Build Evaluation Samples Manifest
eval_indices = list(range(START_SAMPLE_OFFSET, min(START_SAMPLE_OFFSET + NUM_EVAL_SAMPLES, total_available)))
rng = random.Random(RANDOM_SEED)

eval_manifest = []
for s_idx in eval_indices:
    sample_data = benchmark_dataset[s_idx]
    q = sample_data.get("question", "")
    gt = resolve_gt_letter(sample_data.get("answer", ""), sample_data.get("cycle_category", 0))
    opts = sample_data.get("options", {})
    t_rel = sample_data.get("input_image", "")
    
    # Sample Distractors
    min_w = max(0, s_idx - DISTRACTOR_WINDOW)
    max_w = min(total_available - 1, s_idx + DISTRACTOR_WINDOW)
    cand_distractors = [i for i in range(min_w, max_w + 1) if i != s_idx]
    if len(cand_distractors) < (CANDIDATE_POOL_SIZE - 1):
        cand_distractors = [i for i in range(total_available) if i != s_idx]
    
    dist_indices = rng.sample(cand_distractors, CANDIDATE_POOL_SIZE - 1)
    
    eval_manifest.append({
        "sample_index": s_idx,
        "question": q,
        "gt_letter": gt,
        "options": opts,
        "target_path": t_rel,
        "distractor_indices": dist_indices,
        "category": sample_data.get("category", "general"),
        "question_type": "relational" if any(w in q.lower() for w in ["left", "right", "above", "below", "near", "next to", "between"]) else "attribute"
    })

print("=" * 75)
print(f"[Evaluation Manifest] Prepared {len(eval_manifest)} Evaluation Samples (Seed={RANDOM_SEED}):")
for i, m in enumerate(eval_manifest[:5]):
    print(f"  - [{i+1:02d}/{len(eval_manifest):02d}] Sample #{m['sample_index']:03d} | GT: '{m['gt_letter']}' | Type: {m['question_type']} | Q: '{m['question'][:50]}...'")
if len(eval_manifest) > 5:
    print(f"  ... and {len(eval_manifest) - 5} more samples.")
print("=" * 75)

In [ ]:
# ==============================================================================
# CELL 3: MODEL LOADING (QWEN2.5-VL-7B, SIGLIP ViT-SO400M & SAM3)
# ==============================================================================
from mssearch.models.modeling_qwenvl import ModelQwenVL
from mssearch.models.modeling_siglip import SigLIPScorer
try:
    from mssearch.models.modeling_sam3 import sam3_inference
except Exception as e:
    sam3_inference = None
    print(f"[SAM3 Warning] SAM3 import fallback: {e}")

# Checkpoint candidate directories
QWEN_CANDIDATE_PATHS = [
    "/kaggle/input/qwen2-5-vl-7b-instruct",
    "/kaggle/input/qwen2.5-vl-7b-instruct",
    "/kaggle/input/qwen-2-5-vl-7b-instruct",
    "/kaggle/input/qwen2-5/transformers/default/1",
    "Qwen/Qwen2.5-VL-7B-Instruct"
]
qwen_path = next((p for p in QWEN_CANDIDATE_PATHS if os.path.exists(p)), "Qwen/Qwen2.5-VL-7B-Instruct")

SIGLIP_CANDIDATE_PATHS = [
    "/kaggle/input/siglip-so400m-patch14-384",
    "/kaggle/input/siglip/transformers/default/1",
    "/kaggle/input/google/siglip-so400m-patch14-384",
    "google/siglip-so400m-patch14-384"
]
siglip_path = next((p for p in SIGLIP_CANDIDATE_PATHS if os.path.exists(p)), "google/siglip-so400m-patch14-384")

print(f"[Model Discovery] Qwen2.5-VL Path: {qwen_path}")
print(f"[Model Discovery] SigLIP Path:     {siglip_path}")

# Initialize SigLIP Scorer
siglip_engine = SigLIPScorer(
    model_path=siglip_path,
    device=str(device_vlm)
)

# Initialize Qwen2.5-VL Engine
qwen_engine = ModelQwenVL(
    model_path=qwen_path,
    device=str(device_vlm)
)

# Universal Helper Methods for Qwen Engine
if not hasattr(qwen_engine, "get_response"):
    def _universal_get_response(image=None, prompt=None, max_new_tokens=128, **kwargs):
        if image is not None:
            return qwen_engine.generate_response(image_pil=image, question=prompt, max_new_tokens=max_new_tokens)
        return qwen_engine.generate_text(prompt=prompt, max_new_tokens=max_new_tokens)
    qwen_engine.get_response = _universal_get_response

print("=" * 75)
print("[Model Initialization] Qwen2.5-VL and SigLIP engines loaded successfully.")
print("=" * 75)

In [ ]:
# ==============================================================================
# CELL 4: ALGORITHMIC ENGINES: OLD CVSSEARCH vs. NEXT-GEN REDESIGN
# ==============================================================================
from scipy.ndimage import sobel
from sklearn.metrics.pairwise import cosine_similarity
import networkx as nx

# ------------------------------------------------------------------------------
# 1. Direct Canvas Baseline Engine (Old Stage 2 Direct Answering)
# ------------------------------------------------------------------------------
def run_direct_canvas_baseline(
    image: Image.Image,
    question: str,
    options_str: str,
    vlm_model: Any
) -> Dict[str, Any]:
    t0 = time.perf_counter()
    canvas = image.copy()
    if max(canvas.width, canvas.height) > 1024:
        canvas.thumbnail((1024, 1024), Image.Resampling.LANCZOS)
    
    prompt = f"""Inspect this visual region to answer the question:

Question: {question}
Options:
{options_str}

Please carefully observe the visual details.
Provide your final answer enclosed in <answer> tags, for example: <answer>(A)</answer>.
If unanswerable, output <answer>None</answer>.
"""
    resp = vlm_model.get_response(image=canvas, prompt=prompt, max_new_tokens=128)
    latency = time.perf_counter() - t0
    m = re.search(r"<answer>\s*\(?([A-Za-z])[\).:\s]?", str(resp))
    pred = m.group(1).upper() if m else "N/A"
    return {
        "pred": pred,
        "response": resp.strip(),
        "latency": latency,
        "input_size": (canvas.width, canvas.height),
        "regime": "DirectCanvas"
    }

# ------------------------------------------------------------------------------
# 2. Old CVSearch Baseline Engine (Rigid Quadtree + 0.25 Binary Pruning Cliff)
# ------------------------------------------------------------------------------
def run_old_cvsearch_baseline(
    candidate_images: List[Image.Image],
    target_pool_idx: int,
    question: str,
    options_str: str,
    siglip_model: Any,
    vlm_model: Any,
    prune_threshold: float = 0.25,
    max_depth: int = 2
) -> Dict[str, Any]:
    t0 = time.perf_counter()
    img = candidate_images[target_pool_idx]
    w, h = img.size
    
    # Text encoding for query
    text_feat = siglip_model.encode_text([question])
    text_feat = text_feat / (np.linalg.norm(text_feat, axis=-1, keepdims=True) + 1e-8)
    
    nodes = [(0, 0, w, h, 0)] # x1, y1, x2, y2, depth
    surviving_nodes = []
    pruning_cliff_triggered = False
    
    while nodes:
        x1, y1, x2, y2, depth = nodes.pop(0)
        bw, bh = x2 - x1, y2 - y1
        crop = img.crop((x1, y1, x2, y2))
        
        # Global average-pooled cosine similarity (Old Design)
        img_feat = siglip_model.encode_image(crop)
        img_feat = img_feat / (np.linalg.norm(img_feat, axis=-1, keepdims=True) + 1e-8)
        sim = float(np.dot(img_feat, text_feat.T)[0, 0])
        
        # Binary Pruning Cliff
        if sim < prune_threshold:
            continue
            
        if depth >= max_depth or max(bw, bh) <= 768:
            surviving_nodes.append(((x1, y1, x2, y2), sim))
        else:
            # Rigid 2x2 Quadrant Bisection (Old CVSearch)
            mx, my = x1 + bw // 2, y1 + bh // 2
            nodes.append((x1, y1, mx, my, depth + 1))
            nodes.append((mx, y1, x2, my, depth + 1))
            nodes.append((x1, my, mx, y2, depth + 1))
            nodes.append((mx, my, x2, y2, depth + 1))
            
    if not surviving_nodes:
        pruning_cliff_triggered = True
        latency = time.perf_counter() - t0
        return {
            "pred": "N/A",
            "response": "PRUNED_AT_THRESHOLD",
            "latency": latency,
            "pruning_cliff": True,
            "best_crop": None,
            "regime": "OldCVSearchQuadtree"
        }
        
    # Pick top surviving node
    best_box, best_sim = max(surviving_nodes, key=lambda x: x[1])
    eval_crop = img.crop(best_box)
    if max(eval_crop.width, eval_crop.height) > 1024:
        eval_crop.thumbnail((1024, 1024), Image.Resampling.LANCZOS)
        
    prompt = f"""Inspect this visual region to answer the question:

Question: {question}
Options:
{options_str}

Please carefully observe the visual details.
Provide your final answer enclosed in <answer> tags, for example: <answer>(A)</answer>.
If unanswerable, output <answer>None</answer>.
"""
    resp = vlm_model.get_response(image=eval_crop, prompt=prompt, max_new_tokens=128)
    latency = time.perf_counter() - t0
    m = re.search(r"<answer>\s*\(?([A-Za-z])[\).:\s]?", str(resp))
    pred = m.group(1).upper() if m else "N/A"
    return {
        "pred": pred,
        "response": resp.strip(),
        "latency": latency,
        "pruning_cliff": False,
        "best_crop": eval_crop,
        "best_box": best_box,
        "best_sim": best_sim,
        "regime": "OldCVSearchQuadtree"
    }

# ------------------------------------------------------------------------------
# 3. Next-Gen Redesign Engine (Seams + Dense Token Grid + Bayesian Macro Envelopes)
# ------------------------------------------------------------------------------
@dataclass
class RedesignPatch:
    img_idx: int
    box: Tuple[int, int, int, int]
    s_dense: float
    s_pool: float
    entropy: float
    feat_vector: np.ndarray

def find_minimum_energy_elastic_seam(img: Image.Image, box: Tuple[int, int, int, int], axis: str = 'x') -> Tuple[int, int]:
    bx1, by1, bx2, by2 = box
    bw, bh = bx2 - bx1, by2 - by1
    crop = img.crop(box).convert("L")
    crop_gray = np.array(crop, dtype=np.float32)
    corridor_ratio = 0.35
    if axis == 'x':
        grad = np.abs(sobel(crop_gray, axis=1))
        energy_1d = np.sum(grad, axis=0)
        c_min = int(bw * (0.5 - corridor_ratio / 2.0))
        c_max = int(bw * (0.5 + corridor_ratio / 2.0))
        rel_seam = c_min + int(np.argmin(energy_1d[c_min:c_max])) if c_max > c_min else bw // 2
        return bx1 + rel_seam, max(32, min(96, int(bw * 0.15)))
    else:
        grad = np.abs(sobel(crop_gray, axis=0))
        energy_1d = np.sum(grad, axis=1)
        c_min = int(bh * (0.5 - corridor_ratio / 2.0))
        c_max = int(bh * (0.5 + corridor_ratio / 2.0))
        rel_seam = c_min + int(np.argmin(energy_1d[c_min:c_max])) if c_max > c_min else bh // 2
        return by1 + rel_seam, max(32, min(96, int(bh * 0.15)))

def extract_dense_spatial_token_saliency(crop: Image.Image, siglip_model: Any, text_queries: List[str]) -> Tuple[float, float]:
    try:
        t_feat = siglip_model.encode_text(text_queries)
        t_feat = t_feat / (np.linalg.norm(t_feat, axis=-1, keepdims=True) + 1e-8)
        
        # Dense pre-pooling token extraction
        inputs = siglip_model.processor(images=crop, return_tensors="pt").to(siglip_model.device)
        with torch.no_grad():
            vision_out = siglip_model.model.vision_model(**inputs)
            spatial_tokens = vision_out.last_hidden_state # [1, 576, D]
            tokens_norm = spatial_tokens / (spatial_tokens.norm(dim=-1, keepdim=True) + 1e-8)
            t_tensor = torch.from_numpy(t_feat).float().to(siglip_model.device)
            sim_grid = torch.matmul(tokens_norm, t_tensor.T).squeeze(-1).squeeze(0) # [576]
            s_dense = float(sim_grid.max().item())
            s_pool = float(sim_grid.mean().item())
            return s_dense, s_pool
    except Exception:
        # Fallback to standard pooled score
        feat = siglip_model.encode_image(crop)
        t_feat = siglip_model.encode_text(text_queries)
        sim = float(np.dot(feat, t_feat.T)[0, 0])
        return sim, sim

def compute_patch_entropy(crop: Image.Image) -> float:
    gray = crop.convert("L").resize((128, 128))
    hist = gray.histogram()
    total = sum(hist)
    if total == 0: return 0.0
    p = [h / total for h in hist if h > 0]
    return float(-sum(x * np.log2(x) for x in p) / 8.0)

def run_nextgen_redesign_pipeline(
    candidate_images: List[Image.Image],
    target_pool_idx: int,
    question: str,
    options_str: str,
    siglip_model: Any,
    vlm_model: Any,
    max_depth: int = 2
) -> Dict[str, Any]:
    t0 = time.perf_counter()
    
    # Stage 1: Dual-Granularity Semantic Decomposition
    parse_prompt = f"""Extract physical targets from this question as JSON:
Question: "{question}"
Return JSON: {{"atomic_visual_anchors": ["..."], "dense_semantic_queries": ["..."]}}"""
    try:
        raw_parse = vlm_model.get_response(prompt=parse_prompt, max_new_tokens=128)
        cleaned = re.sub(r'```json\s*', '', raw_parse)
        cleaned = re.sub(r'```', '', cleaned).strip()
        data = json.loads(cleaned)
        anchors = data.get("atomic_visual_anchors", ["target"])
        dense_q = data.get("dense_semantic_queries", [question])
    except Exception:
        anchors = ["target"]
        dense_q = [question]
        
    # Stage 3: Sobel Seam Carving + Dense Token Grid Partitioning on Target Canvas
    target_img = candidate_images[target_pool_idx]
    tw, th = target_img.size
    
    patches: List[RedesignPatch] = []
    queue = [(0, 0, tw, th, 0)]
    
    while queue:
        bx1, by1, bx2, by2, depth = queue.pop(0)
        bw, bh = bx2 - bx1, by2 - by1
        crop = target_img.crop((bx1, by1, bx2, by2))
        
        ent = compute_patch_entropy(crop)
        s_dense, s_pool = extract_dense_spatial_token_saliency(crop, siglip_model, dense_q)
        feat = siglip_model.encode_image(crop)
        
        if ent < 0.25 and max(bw, bh) <= 1536:
            continue
            
        if depth >= max_depth or max(bw, bh) <= 512:
            patches.append(RedesignPatch(target_pool_idx, (bx1, by1, bx2, by2), s_dense, s_pool, ent, feat))
        else:
            seam_x, pad_x = find_minimum_energy_elastic_seam(target_img, (bx1, by1, bx2, by2), axis='x')
            seam_y, pad_y = find_minimum_energy_elastic_seam(target_img, (bx1, by1, bx2, by2), axis='y')
            queue.append((bx1, by1, min(bx2, seam_x + pad_x), min(by2, seam_y + pad_y), depth + 1))
            queue.append((max(bx1, seam_x - pad_x), by1, bx2, min(by2, seam_y + pad_y), depth + 1))
            queue.append((bx1, max(by1, seam_y - pad_y), min(bx2, seam_x + pad_x), by2, depth + 1))
            queue.append((max(bx1, seam_x - pad_x), max(by1, seam_y - pad_y), bx2, by2, depth + 1))
            
    if not patches:
        patches.append(RedesignPatch(target_pool_idx, (0, 0, tw, th), 0.5, 0.5, 0.5, siglip_model.encode_image(target_img)))
        
    # Stage 4: Continuous Bayesian Consolidation & Graph Connected Envelopes
    G = nx.Graph()
    for i in range(len(patches)):
        G.add_node(i)
        
    feat_mat = np.array([p.feat_vector for p in patches])
    sim_mat = cosine_similarity(feat_mat)
    
    for i in range(len(patches)):
        for j in range(i + 1, len(patches)):
            bi, bj = patches[i].box, patches[j].box
            dx = max(0, max(bi[0], bj[0]) - min(bi[2], bj[2]))
            dy = max(0, max(bi[1], bj[1]) - min(bi[3], bj[3]))
            dist = np.sqrt(dx**2 + dy**2)
            if dist <= 64 and sim_mat[i, j] >= 0.75:
                G.add_edge(i, j)
                
    components = []
    for comp in nx.connected_components(G):
        m_patches = [patches[idx] for idx in comp]
        ux1 = min(p.box[0] for p in m_patches)
        uy1 = min(p.box[1] for p in m_patches)
        ux2 = max(p.box[2] for p in m_patches)
        uy2 = max(p.box[3] for p in m_patches)
        macro_env = (ux1, uy1, ux2, uy2)
        
        # Argmax Evidential Peak (p*)
        rep_p = max(m_patches, key=lambda p: p.s_dense)
        e_prior = rep_p.s_dense
        
        # Micro-fovea confidence check on atomic noun
        fovea_crop = target_img.crop(rep_p.box)
        try:
            c_vlm = float(vlm_model.get_confidence_value(image=fovea_crop, active_object=anchors[0]))
        except Exception:
            c_vlm = 0.5
            
        e_post = e_prior + 0.35 * c_vlm
        components.append({
            "envelope": macro_env,
            "rep_box": rep_p.box,
            "e_post": e_post,
            "patches_count": len(m_patches)
        })
        
    best_comp = max(components, key=lambda c: c["e_post"])
    macro_box = best_comp["envelope"]
    peak_box = best_comp["rep_box"]
    
    # Stage 6 Reading Crops
    # 1. Macro Holistic Context Crop
    macro_crop = target_img.crop(macro_box)
    if max(macro_crop.width, macro_crop.height) > 768:
        macro_crop.thumbnail((768, 768), Image.Resampling.LANCZOS)
        
    # 2. Native 1:1 Peak Detail Crop
    peak_crop = target_img.crop(peak_box)
    
    reading_prompt = f"""Inspect this visual region to answer the question:

Question: {question}
Options:
{options_str}

Please carefully observe the visual details.
Provide your final answer enclosed in <answer> tags, for example: <answer>(A)</answer>.
If unanswerable, output <answer>None</answer>.
"""
    resp_macro = vlm_model.get_response(image=macro_crop, prompt=reading_prompt, max_new_tokens=128)
    m_macro = re.search(r"<answer>\s*\(?([A-Za-z])[\).:\s]?", str(resp_macro))
    pred_macro = m_macro.group(1).upper() if m_macro else "N/A"
    
    resp_peak = vlm_model.get_response(image=peak_crop, prompt=reading_prompt, max_new_tokens=128)
    m_peak = re.search(r"<answer>\s*\(?([A-Za-z])[\).:\s]?", str(resp_peak))
    pred_peak = m_peak.group(1).upper() if m_peak else "N/A"
    
    # Unified Ensemble Decision
    pred_unified = pred_peak if pred_peak != "N/A" else pred_macro
    latency = time.perf_counter() - t0
    
    return {
        "pred_macro": pred_macro,
        "resp_macro": resp_macro.strip(),
        "pred_peak": pred_peak,
        "resp_peak": resp_peak.strip(),
        "pred_unified": pred_unified,
        "latency": latency,
        "macro_crop": macro_crop,
        "macro_box": macro_box,
        "peak_crop": peak_crop,
        "peak_box": peak_box,
        "e_post": best_comp["e_post"],
        "num_components": len(components),
        "regime": "NextGenRedesign"
    }

In [ ]:
# ==============================================================================
# CELL 5: DETAILED PER-SAMPLE VISUAL DIAGNOSTIC CARD GENERATOR
# ==============================================================================
def generate_sample_diagnostic_card(
    sample_meta: Dict[str, Any],
    direct_res: Dict[str, Any],
    old_res: Dict[str, Any],
    redesign_res: Dict[str, Any],
    target_img: Image.Image,
    save_path: str
):
    """
    Constructs and saves a publication-grade 4-panel visual inspection card:
      Panel 1: Candidate Full Canvas with Target Bounding Box Overlays
      Panel 2: Old CVSearch Quadtree Bisection & Binary Cliff Status
      Panel 3: Next-Gen Redesign Macro Holistic Context Envelope
      Panel 4: Next-Gen Redesign Native 1:1 Peak Detail Crop & Decision Matrix
    """
    fig, axes = plt.subplots(1, 4, figsize=(24, 6))
    gt = sample_meta["gt_letter"]
    q = sample_meta["question"]
    s_idx = sample_meta["sample_index"]
    
    # -------------------------------------------------------------------------
    # Panel 1: Full Canvas Overview
    # -------------------------------------------------------------------------
    thumb = target_img.copy()
    thumb.thumbnail((1024, 1024), Image.Resampling.LANCZOS)
    axes[0].imshow(thumb)
    axes[0].set_title(f"1. Target Canvas (Sample #{s_idx:03d})\nOrig Size: {target_img.width}x{target_img.height}px", fontsize=11, fontweight="bold")
    axes[0].axis("off")
    
    # -------------------------------------------------------------------------
    # Panel 2: Old CVSearch Quadtree
    # -------------------------------------------------------------------------
    old_pred = old_res.get("pred", "N/A")
    old_correct = (old_pred == gt)
    old_color = "forestgreen" if old_correct else "crimson"
    
    if old_res.get("best_crop") is not None:
        axes[1].imshow(old_res["best_crop"])
        axes[1].set_title(f"2. Old CVSearch Quadtree\nPred: '{old_pred}' (GT: '{gt}') | Lat: {old_res['latency']:.2f}s",
                          fontsize=11, fontweight="bold", color=old_color)
    else:
        axes[1].text(0.5, 0.5, f"PRUNED AT CLIFF\n(Score < 0.25)\nPred: '{old_pred}'\nCorrect: False",
                     ha="center", va="center", fontsize=13, fontweight="bold", color="crimson")
        axes[1].set_title(f"2. Old CVSearch Quadtree\nBinary Pruning Cliff (<0.25)", fontsize=11, fontweight="bold", color="crimson")
    axes[1].axis("off")
    for spine in axes[1].spines.values():
        spine.set_edgecolor(old_color)
        spine.set_linewidth(3)
        
    # -------------------------------------------------------------------------
    # Panel 3: Redesign Macro Holistic Context
    # -------------------------------------------------------------------------
    macro_pred = redesign_res.get("pred_macro", "N/A")
    macro_correct = (macro_pred == gt)
    macro_color = "forestgreen" if macro_correct else "crimson"
    axes[2].imshow(redesign_res["macro_crop"])
    axes[2].set_title(f"3. Redesign Macro Context\nPred: '{macro_pred}' (GT: '{gt}') | Post: {redesign_res['e_post']:.2f}",
                      fontsize=11, fontweight="bold", color=macro_color)
    axes[2].axis("off")
    for spine in axes[2].spines.values():
        spine.set_edgecolor(macro_color)
        spine.set_linewidth(3)
        
    # -------------------------------------------------------------------------
    # Panel 4: Redesign Native 1:1 Peak Detail
    # -------------------------------------------------------------------------
    peak_pred = redesign_res.get("pred_peak", "N/A")
    peak_correct = (peak_pred == gt)
    peak_color = "forestgreen" if peak_correct else "crimson"
    axes[3].imshow(redesign_res["peak_crop"])
    axes[3].set_title(f"4. Redesign Native 1:1 Peak Crop\nPred: '{peak_pred}' (GT: '{gt}') | Lat: {redesign_res['latency']:.2f}s",
                      fontsize=11, fontweight="bold", color=peak_color)
    axes[3].axis("off")
    for spine in axes[3].spines.values():
        spine.set_edgecolor(peak_color)
        spine.set_linewidth(3)
        
    plt.suptitle(f"Sample #{s_idx:03d} Diagnostic Comparison | Question: '{q}'\nGround Truth: Option '{gt}'",
                 fontsize=13, fontweight="bold", color="#1a237e")
    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.close()

In [ ]:
# ==============================================================================
# CELL 6: BATCH EVALUATION EXECUTION LOOP (N=30 SAMPLES)
# ==============================================================================
benchmark_results = []
checkpoint_file = os.path.join(TELEMETRY_DIR, "checkpoint_results.jsonl")

print("=" * 85)
print(f"STARTING COMPREHENSIVE MULTI-SAMPLE EVALUATION: N = {len(eval_manifest)} SAMPLES")
print("=" * 85)

for i, sample in enumerate(eval_manifest):
    s_idx = sample["sample_index"]
    q = sample["question"]
    gt = sample["gt_letter"]
    opts_str = format_options_str(sample["options"])
    
    t_start_sample = time.perf_counter()
    print(f"\n--- [{i+1:02d}/{len(eval_manifest):02d}] Processing Sample #{s_idx:03d} | Q: '{q[:50]}...' | GT: '{gt}' ---")
    
    # 1. Load Candidate Pool (1 Target + N-1 Distractors)
    t_rel = sample["target_path"]
    t_path = os.path.join(HRBENCH_BASE_DIR, t_rel) if not os.path.exists(t_rel) else t_rel
    target_canvas = Image.open(t_path).convert("RGB")
    
    pool_images = [target_canvas]
    for d_idx in sample["distractor_indices"]:
        d_rel = benchmark_dataset[d_idx].get("input_image", "")
        d_path = os.path.join(HRBENCH_BASE_DIR, d_rel) if not os.path.exists(d_rel) else d_rel
        pool_images.append(Image.open(d_path).convert("RGB"))
        
    # Target is placed at pool index 0 for controlled evaluation
    target_pool_idx = 0
    
    # 2. Run Direct Canvas Baseline (Old Stage 2)
    res_direct = run_direct_canvas_baseline(target_canvas, q, opts_str, qwen_engine)
    
    # 3. Run Old CVSearch Quadtree Baseline
    res_old = run_old_cvsearch_baseline(pool_images, target_pool_idx, q, opts_str, siglip_engine, qwen_engine)
    
    # 4. Run Next-Gen Redesign Pipeline
    res_redesign = run_nextgen_redesign_pipeline(pool_images, target_pool_idx, q, opts_str, siglip_engine, qwen_engine)
    
    # Evaluation Correctness Booleans
    acc_direct = (res_direct["pred"] == gt)
    acc_old = (res_old["pred"] == gt)
    acc_macro = (res_redesign["pred_macro"] == gt)
    acc_peak = (res_redesign["pred_peak"] == gt)
    acc_unified = (res_redesign["pred_unified"] == gt)
    
    # Generate & Save Per-Sample Diagnostic Visualization Card
    vis_path = os.path.join(VIS_DIR, f"sample_{s_idx:03d}_diagnostic.png")
    generate_sample_diagnostic_card(sample, res_direct, res_old, res_redesign, target_canvas, vis_path)
    
    # Record Telemetry Metrics Record
    rec = {
        "sample_index": s_idx,
        "question": q,
        "ground_truth": gt,
        "question_type": sample["question_type"],
        "pred_direct": res_direct["pred"],
        "acc_direct": acc_direct,
        "lat_direct": res_direct["latency"],
        "pred_old": res_old["pred"],
        "acc_old": acc_old,
        "lat_old": res_old["latency"],
        "old_pruned": res_old["pruning_cliff"],
        "pred_macro": res_redesign["pred_macro"],
        "acc_macro": acc_macro,
        "pred_peak": res_redesign["pred_peak"],
        "acc_peak": acc_peak,
        "pred_unified": res_redesign["pred_unified"],
        "acc_unified": acc_unified,
        "lat_redesign": res_redesign["latency"],
        "e_post": res_redesign["e_post"],
        "vis_card_path": vis_path
    }
    benchmark_results.append(rec)
    
    # Append to streaming checkpoint file
    with open(checkpoint_file, "a", encoding="utf-8") as f:
        f.write(json.dumps(rec) + "\n")
        
    # Print Live Progress 1-Liner
    print(f"  [Sample #{s_idx:03d} Result] GT: '{gt}' | Direct: '{res_direct['pred']}' ({acc_direct}) | Old CVS: '{res_old['pred']}' ({acc_old}) | Redesign Peak: '{res_redesign['pred_peak']}' ({acc_peak}) | Card Saved.")

print("\n" + "=" * 85)
print(f"EVALUATION COMPLETE: Processed {len(benchmark_results)} samples. Checkpoint: {checkpoint_file}")
print("=" * 85)

In [ ]:
# ==============================================================================
# CELL 7: GLOBAL ACCURACY SYNTHESIS & STATISTICAL REPORTING
# ==============================================================================
total_n = len(benchmark_results)

# Overall Accuracies
acc_direct_pct = sum(r["acc_direct"] for r in benchmark_results) / total_n * 100.0
acc_old_pct = sum(r["acc_old"] for r in benchmark_results) / total_n * 100.0
acc_macro_pct = sum(r["acc_macro"] for r in benchmark_results) / total_n * 100.0
acc_peak_pct = sum(r["acc_peak"] for r in benchmark_results) / total_n * 100.0
acc_unified_pct = sum(r["acc_unified"] for r in benchmark_results) / total_n * 100.0

# Pruning Cliff Statistics
pruning_cliff_rate = sum(r["old_pruned"] for r in benchmark_results) / total_n * 100.0

# Subcategory Splits
attr_samples = [r for r in benchmark_results if r["question_type"] == "attribute"]
rel_samples = [r for r in benchmark_results if r["question_type"] == "relational"]

def get_cat_acc(sample_list, key):
    return (sum(r[key] for r in sample_list) / len(sample_list) * 100.0) if sample_list else 0.0

# Latencies
avg_lat_direct = sum(r["lat_direct"] for r in benchmark_results) / total_n
avg_lat_old = sum(r["lat_old"] for r in benchmark_results) / total_n
avg_lat_redesign = sum(r["lat_redesign"] for r in benchmark_results) / total_n

print("=" * 88)
print("              COMPREHENSIVE STATISTICAL BENCHMARK REPORT (N = %d)" % total_n)
print("=" * 88)
print(f"{'REGIME / EVALUATION ENGINE':<32} | {'OVERALL ACC':<12} | {'ATTRIBUTE':<12} | {'RELATIONAL':<12} | {'AVG LATENCY':<12}")
print("-" * 88)
print(f"{'1. Direct Canvas (Old Stage 2)':<32} | {acc_direct_pct:>10.1f}% | {get_cat_acc(attr_samples, 'acc_direct'):>10.1f}% | {get_cat_acc(rel_samples, 'acc_direct'):>10.1f}% | {avg_lat_direct:>10.2f}s")
print(f"{'2. Old CVSearch (Quadtree)':<32} | {acc_old_pct:>10.1f}% | {get_cat_acc(attr_samples, 'acc_old'):>10.1f}% | {get_cat_acc(rel_samples, 'acc_old'):>10.1f}% | {avg_lat_old:>10.2f}s")
print(f"{'3. Redesign Macro Context':<32} | {acc_macro_pct:>10.1f}% | {get_cat_acc(attr_samples, 'acc_macro'):>10.1f}% | {get_cat_acc(rel_samples, 'acc_macro'):>10.1f}% | {avg_lat_redesign:>10.2f}s")
print(f"{'4. Redesign Native 1:1 Peak':<32} | {acc_peak_pct:>10.1f}% | {get_cat_acc(attr_samples, 'acc_peak'):>10.1f}% | {get_cat_acc(rel_samples, 'acc_peak'):>10.1f}% | {avg_lat_redesign:>10.2f}s")
print(f"{'5. Redesign Unified Consensus':<32} | {acc_unified_pct:>10.1f}% | {get_cat_acc(attr_samples, 'acc_unified'):>10.1f}% | {get_cat_acc(rel_samples, 'acc_unified'):>10.1f}% | {avg_lat_redesign:>10.2f}s")
print("=" * 88)
print(f"[Key Failure Diagnostic] Old CVSearch Pruning Cliff Rate (<0.25): {pruning_cliff_rate:.1f}% of samples completely dropped!")
print(f"[Key Failure Diagnostic] Next-Gen Redesign Pruning Cliff Rate:   0.0% (Zero binary dropouts).")
print("=" * 88)

# Save Master Summary JSON
summary_report = {
    "total_samples": total_n,
    "accuracies": {
        "direct_canvas": acc_direct_pct,
        "old_cvsearch_quadtree": acc_old_pct,
        "redesign_macro_context": acc_macro_pct,
        "redesign_native_peak": acc_peak_pct,
        "redesign_unified": acc_unified_pct
    },
    "category_breakdown": {
        "attribute_count": len(attr_samples),
        "relational_count": len(rel_samples),
        "attribute_acc_peak": get_cat_acc(attr_samples, "acc_peak"),
        "relational_acc_macro": get_cat_acc(rel_samples, "acc_macro"),
        "relational_acc_peak": get_cat_acc(rel_samples, "acc_peak")
    },
    "pruning_cliff_rate_old": pruning_cliff_rate,
    "avg_latencies": {
        "direct_canvas": avg_lat_direct,
        "old_cvsearch": avg_lat_old,
        "redesign": avg_lat_redesign
    },
    "samples_telemetry": benchmark_results
}

summary_json_path = os.path.join(OUTPUT_DIR, "summary_metrics.json")
with open(summary_json_path, "w", encoding="utf-8") as f:
    json.dump(summary_report, f, indent=2)

print(f"[Report Export] Master summary JSON saved to: {summary_json_path}")

In [ ]:
# ==============================================================================
# CELL 8: PUBLICATION-GRADE SUMMARY VISUALIZATIONS & ARTIFACT EXPORT
# ==============================================================================
# 1. Summary Accuracy Comparison Bar Chart
fig, ax = plt.subplots(figsize=(10, 6))
regimes = ["Direct Canvas", "Old CVSearch", "Redesign Macro", "Redesign 1:1 Peak", "Redesign Unified"]
accs = [acc_direct_pct, acc_old_pct, acc_macro_pct, acc_peak_pct, acc_unified_pct]
bar_colors = ["#78909c", "#e57373", "#64b5f6", "#81c784", "#2e7d32"]

bars = ax.bar(regimes, accs, color=bar_colors, edgecolor="black", linewidth=1.5, width=0.6)
ax.set_ylim(0, 105)
ax.set_ylabel("Accuracy (%)", fontsize=12, fontweight="bold")
ax.set_title(f"Multi-Regime Accuracy Comparison on HRBench (N = {total_n})", fontsize=14, fontweight="bold")

for bar in bars:
    yval = bar.get_height()
    ax.text(bar.get_x() + bar.get_width() / 2.0, yval + 1.5, f"{yval:.1f}%", ha="center", va="bottom", fontsize=11, fontweight="bold")

ax.grid(axis="y", linestyle="--", alpha=0.5)
plt.tight_layout()
acc_chart_path = os.path.join(OUTPUT_DIR, "accuracy_comparison_bar.png")
plt.savefig(acc_chart_path, dpi=200)
plt.show()

# 2. Pruning Cliff & Failure Mode Pie Chart
fig, ax = plt.subplots(figsize=(7, 5))
labels = ["Old CVSearch: Pruned at Cliff (<0.25)", "Old CVSearch: Evaluated"]
sizes = [pruning_cliff_rate, max(0.0, 100.0 - pruning_cliff_rate)]
colors = ["#d32f2f", "#4caf50"]
ax.pie(sizes, labels=labels, autopct="%1.1f%%", startangle=140, colors=colors,
       wedgeprops={"edgecolor": "black", "linewidth": 1.5})
ax.set_title("Old CVSearch Search Cliff Breakdown", fontsize=12, fontweight="bold")
plt.tight_layout()
cliff_chart_path = os.path.join(OUTPUT_DIR, "pruning_cliff_breakdown.png")
plt.savefig(cliff_chart_path, dpi=200)
plt.show()

print("=" * 75)
print("EVALUATION ARTIFACTS READY FOR INSPECTION:")
print(f"  - Summary Metrics JSON:       {summary_json_path}")
print(f"  - Accuracy Comparison Chart:  {acc_chart_path}")
print(f"  - Pruning Cliff Breakdown:    {cliff_chart_path}")
print(f"  - Per-Sample Diagnostic Cards: {VIS_DIR} (Total: {total_n} PNG files)")
print("=" * 75)